In [ ]:
import os
import sys
from pathlib import Path

# Required for the repository's deterministic CUDA training.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
import torchvision

print("Python:", sys.version)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda if torch.version.cuda else "None")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Working directory:", Path.cwd())


# EXP-DN121-BASE-S42 — Kaggle orchestration

This notebook delegates model, data-pipeline, metrics, checkpointing, and training work to the repository script exactly as committed. The first DenseNet baseline run uses exactly commit `0daf10aa0d6a19bd53ecc2eb8d24e6b1b3d19db2`, containing the validated --image-root implementation. No runtime patching of tracked source code is allowed. Test images are checked for file presence only, never decoded or evaluated.

Enable a GPU, enable Internet for GitHub and official ImageNet weights, and attach the frozen image dataset before running all cells. No packages are installed.


In [ ]:
# Confirmed mount for samairashahida/medtrust-nih-224.
# Images are individual files: no extraction or dataset copying is needed.
KAGGLE_IMAGE_ROOT = "/kaggle/input/medtrust-nih-224/images_224x224"
KAGGLE_WORK_ROOT = "/kaggle/working/MedTrust-AI"

REPOSITORY_URL = "https://github.com/Samaira-0/MedTrust-AI.git"
EXPECTED_GIT_COMMIT = "0daf10aa0d6a19bd53ecc2eb8d24e6b1b3d19db2"
EXPERIMENT_ID = "EXP-DN121-BASE-S42"
assert torch.cuda.is_available(), "Enable a Kaggle GPU before training; CPU training is disabled here."


In [ ]:
import subprocess
import re

assert re.fullmatch(r"[0-9a-f]{40}", EXPECTED_GIT_COMMIT), (
    "Set EXPECTED_GIT_COMMIT to the full exact committed Git SHA before cloning or training."
)
work_root = Path(KAGGLE_WORK_ROOT)
image_root = Path(KAGGLE_IMAGE_ROOT)
assert image_root.is_dir(), f"Attach the frozen dataset and correct KAGGLE_IMAGE_ROOT: {image_root}"
if not work_root.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(work_root)], check=True)
else:
    assert (work_root / ".git").is_dir(), "Work root exists but is not a Git repository."
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=work_root, text=True).strip()
    assert origin == REPOSITORY_URL, f"Unexpected repository origin: {origin}"
    assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=work_root, text=True).strip(), (
        "Existing checkout is dirty. Start a fresh Kaggle session; do not discard prior outputs."
    )
subprocess.run(["git", "checkout", "--detach", EXPECTED_GIT_COMMIT], cwd=work_root, check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=work_root, text=True).strip()
assert commit == EXPECTED_GIT_COMMIT, f"Git revision mismatch: {commit} != {EXPECTED_GIT_COMMIT}"
print("Checked-out Git commit:", commit)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=work_root, text=True).strip(), (
    "Checkout must be clean; runtime patching is forbidden."
)
os.chdir(work_root)
import shutil
print("KAGGLE_IMAGE_ROOT:", KAGGLE_IMAGE_ROOT)
print("Free disk space (/kaggle/working), bytes:", shutil.disk_usage(work_root.parent).free)


In [ ]:
# Metadata checks only: do not open/decode test images.
import importlib.util
for module in ("pandas", "numpy", "yaml", "PIL", "sklearn"):
    assert importlib.util.find_spec(module) is not None, f"Missing dependency: {module}; stop, do not install."
sys.dont_write_bytecode = True
sys.path.insert(0, str(work_root / "scripts"))
from train_densenet121_baseline import load_manifest, load_configs, build_pipelines, CohortDataset, main

manifest_path = work_root / "manifests/nih_effusion_vs_nofinding_v1.csv"
assert manifest_path.is_file(), f"Missing frozen manifest: {manifest_path}"
manifest = load_manifest()  # Includes balance, patient overlap, and patient-cap assertions.
assert len(manifest) == 19326
assert manifest["Image Index"].duplicated().sum() == 0
for split, count in {"train": 12384, "val": 3078, "test": 3864}.items():
    assert int(manifest["split"].eq(split).sum()) == count
    print(f"{split}: {count}")
missing, empty = [], []
for name in manifest["Image Index"]:
    assert Path(name).name == name and "/" not in name and "\\" not in name
    path = image_root / name
    if not path.is_file():
        missing.append(name)
    elif path.stat().st_size == 0:
        empty.append(name)
if missing or empty:
    print("Missing images:", missing)
    print("Zero-byte images:", empty)
assert not missing and not empty, "Frozen cohort is incomplete; training must not start."
# Inspect the pinned trainer without executing main or opening any image.
import ast
import inspect
trainer_tree = ast.parse(inspect.getsource(main))
dataset_calls = [node for node in ast.walk(trainer_tree)
                 if isinstance(node, ast.Call) and isinstance(node.func, ast.Name)
                 and node.func.id == "CohortDataset"]
assert len(dataset_calls) == 1, "Unexpected dataset construction; stop before training."
dataset_comprehensions = [node for node in ast.walk(trainer_tree)
                          if isinstance(node, ast.DictComp) and node.value is dataset_calls[0]]
assert len(dataset_comprehensions) == 1
assert len(dataset_comprehensions[0].generators) == 1
assert ast.literal_eval(dataset_comprehensions[0].generators[0].iter) == ("train", "val")
# The constructor must reject test before any image access.
try:
    CohortDataset(manifest.iloc[:0], "test", None, image_root=image_root)
except ValueError as error:
    assert "Test image loading is forbidden" in str(error)
else:
    raise AssertionError("Trainer did not reject test loading; stop before training.")
print("PASS: trainer constructs only train/val datasets and rejects test loading.")
configs = load_configs()
build_pipelines(configs)  # Validate frozen deterministic preprocessing and affine-only augmentation.
training = configs["model"]["training"]
assert training == {
    "seed": 42, "epochs": 20, "batch_size": 32, "optimizer": "AdamW",
    "learning_rate": 0.0001, "weight_decay": 0.0001, "loss": "BCEWithLogitsLoss",
    "class_weights": None,
    "early_stopping": {"enabled": True, "monitor": "val_auroc", "mode": "max", "patience": 5},
    "scheduler": {"type": "ReduceLROnPlateau", "monitor": "val_auroc", "mode": "max", "factor": 0.5, "patience": 2},
}
experiment_dir = work_root / "experiments" / EXPERIMENT_ID
assert not experiment_dir.exists() or not any(experiment_dir.iterdir()), "Existing experiment outputs: use a fresh session."
print("PASS: all 19326 referenced files present and nonzero; frozen configuration verified.")
print("Test images were checked for file presence only; no test image was decoded.")


## Run the repository trainer on GPU

This cell starts full training **on Kaggle**. The repository creates only train and val datasets, uses validation AUROC for scheduler/early stopping/checkpoint selection, and saves predictions only for the best validation checkpoint. The fixed 0.5 threshold is descriptive. No test evaluation or threshold selection occurs.


In [ ]:
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=work_root, text=True).strip() == EXPECTED_GIT_COMMIT
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=work_root, text=True).strip(), (
    "Tracked source must remain exactly as committed before training."
)
command = [sys.executable, "scripts/train_densenet121_baseline.py", "--image-root", str(image_root)]
print("Running:", command)
subprocess.run(command, cwd=work_root, env=os.environ.copy(), check=True)


In [ ]:
import json
import pandas as pd

expected_outputs = ["checkpoint_best.pt", "checkpoint_last.pt", "training_history.csv",
                    "config_snapshot.yaml", "metadata.json", "val_predictions.csv", "val_logits.npy"]
for name in expected_outputs:
    assert (experiment_dir / name).is_file(), f"Missing experiment output: {name}"
metadata_path = experiment_dir / "metadata.json"
metadata = json.loads(metadata_path.read_text())
assert metadata["git_commit"] == EXPECTED_GIT_COMMIT
assert metadata["device"].startswith("cuda"), "Run was not GPU training."
assert metadata["test_images_loaded"] == 0
history = pd.read_csv(experiment_dir / "training_history.csv")
best_rows = history[history["epoch"].eq(metadata["best_epoch"])]
assert len(best_rows) == 1
best_row = best_rows.iloc[0]
print("Best epoch:", metadata["best_epoch"])
print("Best validation AUROC:", metadata["best_val_auroc"])
print("Validation AUPRC (average precision, best epoch):", best_row["val_auprc"])
print("Validation BCE/NLL loss (best epoch):", best_row["val_bce_nll"])
# The history records LR before scheduler.step; use saved optimizer state for final LR.
last_checkpoint = torch.load(experiment_dir / "checkpoint_last.pt", map_location="cpu", weights_only=False)
final_learning_rates = [group["lr"] for group in last_checkpoint["optimizer_state_dict"]["param_groups"]]
assert len(set(final_learning_rates)) == 1
print("Final learning rate (after scheduler):", final_learning_rates[0])
early_stopped = bool(training["early_stopping"]["enabled"] and
                     last_checkpoint["bad_epochs"] >= training["early_stopping"]["patience"])
print("Early stopping occurred:", early_stopped)
del last_checkpoint
assert metadata["git_dirty"] is False, "Training must use a clean committed repository."
assert not subprocess.check_output(["git", "diff", "HEAD", "--"], cwd=work_root, text=True).strip(), (
    "Tracked files changed during the run."
)
print("Experiment directory:", experiment_dir)
for path in sorted(experiment_dir.iterdir()):
    print(path.name, path.stat().st_size, "bytes")


In [ ]:
import shutil
from IPython.display import FileLink, display

archive = shutil.make_archive(
    "/kaggle/working/EXP-DN121-BASE-S42", "zip",
    root_dir=experiment_dir.parent, base_dir=experiment_dir.name,
)
print("Archive:", archive)
print("Archive size:", Path(archive).stat().st_size, "bytes")
display(FileLink(archive))
